In [ ]:
# AI CLOUD RESOURCE OPTIMIZATION SYSTEM - MODEL COMPARISON (RF vs XGBoost)
import pandas as pd
import numpy as np
import time

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

import matplotlib.pyplot as plt
import joblib

# 1. LOAD CLOUD DATASET
df = pd.read_csv("cloud_historical_data.csv")

print("CLOUD RESOURCE DATASET SUMMARY")
print(f"Total records: {df.shape[0]}")
print(df.head())

# 2. FEATURES AND TARGET
# Drop timestamp since it's a non-numeric/time feature not suitable directly for tree models
X = df.drop(columns=["required_servers", "timestamp"])
y = df["required_servers"]

# 3. TRAIN TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(f"\nTraining Samples: {len(X_train)}")
print(f"Testing Samples: {len(X_test)}")

# 4. MODEL TRAINING AND TIMING

# A. Random Forest Regressor
print("\nTraining Random Forest Regressor...")
rf_start_time = time.time()
rf_model = RandomForestRegressor(n_estimators=200, random_state=42)
rf_model.fit(X_train, y_train)
rf_train_time = time.time() - rf_start_time
rf_predictions = rf_model.predict(X_test)

# B. XGBoost Regressor
print("Training XGBoost Regressor...")
xgb_start_time = time.time()
xgb_model = XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=6, random_state=42)
xgb_model.fit(X_train, y_train)
xgb_train_time = time.time() - xgb_start_time
xgb_predictions = xgb_model.predict(X_test)

# 5. MODEL EVALUATION
def evaluate_predictions(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, r2

rf_mae, rf_rmse, rf_r2 = evaluate_predictions(y_test, rf_predictions)
xgb_mae, xgb_rmse, xgb_r2 = evaluate_predictions(y_test, xgb_predictions)

# 6. COMPARISON REPORT
comparison_df = pd.DataFrame({
    "Metric": ["Mean Absolute Error (MAE)", "Root Mean Squared Error (RMSE)", "R2 Score", "Training Time (s)"],
    "Random Forest": [rf_mae, rf_rmse, rf_r2, rf_train_time],
    "XGBoost": [xgb_mae, xgb_rmse, xgb_r2, xgb_train_time]
})

print("\n================ MODEL COMPARISON REPORT ================")
print(comparison_df.to_string(index=False))
print("=========================================================")

# Determine the best model (using R2 Score as primary metric)
if xgb_r2 > rf_r2:
    best_model = xgb_model
    best_name = "XGBoost"
    best_predictions = xgb_predictions
else:
    best_model = rf_model
    best_name = "Random Forest"
    best_predictions = rf_predictions

print(f"\nBest Model Selected: {best_name} (R2: {r2_score(y_test, best_predictions):.4f})")

# 7. FEATURE IMPORTANCE COMPARISON
rf_importance = rf_model.feature_importances_
xgb_importance = xgb_model.feature_importances_
features = X.columns

importance_df = pd.DataFrame({
    "Feature": features,
    "RF Importance": rf_importance,
    "XGBoost Importance": xgb_importance
})
print("\nFEATURE IMPORTANCES:")
print(importance_df.to_string(index=False))

# Plot Feature Importances Side by Side
x_indices = np.arange(len(features))
width = 0.35

plt.figure(figsize=(12, 6))
plt.bar(x_indices - width/2, rf_importance, width, label='Random Forest', color='#1f77b4')
plt.bar(x_indices + width/2, xgb_importance, width, label='XGBoost', color='#ff7f0e')
plt.xlabel('Cloud Features')
plt.ylabel('Importance')
plt.title('Feature Importances: Random Forest vs XGBoost')
plt.xticks(x_indices, features, rotation=15)
plt.legend()
plt.tight_layout()
plt.show()

# 8. ACTUAL VS PREDICTED PLOTS
plt.figure(figsize=(12, 6))
plt.plot(y_test.values[:50], label="Actual Servers", color='black', linewidth=2, linestyle='--')
plt.plot(rf_predictions[:50], label="Random Forest Predictions", color='#1f77b4', alpha=0.8)
plt.plot(xgb_predictions[:50], label="XGBoost Predictions", color='#ff7f0e', alpha=0.8)
plt.xlabel("Test Samples (First 50)")
plt.ylabel("Servers")
plt.title("Model Predictions vs Actual Required Servers")
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()

# 9. SAVE THE BEST PERFORMING MODEL
joblib.dump(best_model, "cloud_resource_optimization_model.pkl")
print(f"\nSUCCESSFULLY SAVED THE BEST MODEL ({best_name}) to cloud_resource_optimization_model.pkl!")

# 10. AUTO-SCALING FUNCTION WITH SELECTED BEST MODEL
def auto_scale(cpu, memory, traffic, users, servers):
    input_data = pd.DataFrame({
        "cpu_usage": [cpu],
        "memory_usage": [memory],
        "network_traffic": [traffic],
        "active_users": [users],
        "current_servers": [servers]
    })
    
    # Load from the saved .pkl file to simulate production behavior
    prod_model = joblib.load("cloud_resource_optimization_model.pkl")
    prediction = prod_model.predict(input_data)
    required = round(prediction[0])
    
    print("\n========== PRODUCTION AUTO SCALING REPORT ==========")
    print(f"Model Type: {type(prod_model).__name__}")
    print("Current Servers:", servers)
    print("Predicted Required Servers:", required)
    
    if required > servers:
        print("ACTION: SCALE UP SERVERS")
    elif required < servers:
        print("ACTION: SCALE DOWN SERVERS")
    else:
        print("ACTION: NO SCALING NEEDED")

# Test Auto-Scaling logic
auto_scale(cpu=92, memory=90, traffic=480, users=380, servers=5)
